# 📘Домашнє завдання №9 Описовий аналіз даних (Exploratory Data Analysis)

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW9

Використати вбудований датасет:

In [1]:
# Download data (to cover the case when the data aren't accessible)

# !pip install --upgrade nbformat
# !pip install jinja2

import shutil
import sqlite3
import pandas as pd
import seaborn as sns
from pathlib import Path

from IPython.core.display import Markdown
from IPython.display import display
from pandas.core.interchange.dataframe_protocol import DataFrame

# Input data
ds_name = "iris"
db_file_name = "store_hw9.db"
git_project_url = "https://github.com/BogdanPinchuk/DataScience-PBY_HW9.git"
main_file_name = "Bohdan_Pinchuk_DS_HW9.ipynb"

# Solution

# Note: to handle error: "SSL: CERTIFICATE_VERIFY_FAILED" or no connection to the server
try:
    # for testing
    # raise Exception
    ds_data = sns.load_dataset(ds_name)

    # Use only one time to initialize/update data (at first time)
    # conn = sqlite3.connect(db_file_name)
    # ds_data.to_sql(ds_name, conn, if_exists="replace", index=False)
    # conn.close()
except Exception:
    file_path = Path(db_file_name)

    if not file_path.exists():
        # upload all files
        current_path = !pwd
        current_path = current_path[0]
        parent_path = !dirname "$current_path"
        parent_path = parent_path[0]
        temp_path = f"{parent_path}/temp"

        # Clone data
        !rm -rf "$temp_path"
        !git clone "$git_project_url" "$temp_path"

        source = Path(temp_path)
        destination = Path(current_path)
        exclude = {main_file_name, ".git", ".idea"}

        for item in source.iterdir():
            if item.name in exclude:
                continue

            target = destination / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)

        # Clean temp folder
        !rm -rf "$temp_path"

    conn = sqlite3.connect(db_file_name)
    ds_data = pd.read_sql(f"SELECT * FROM {ds_name}", conn)
    conn.close()

# display(ds_data)

In [2]:
## Отримання даних

# Solution
iris = ds_data.copy()

# Print results
display(iris)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


Датасет дані про квіти ірисів трьох видів:
* Setosa
* Versicolor
* Virginica

Для кожної квітки виміряно 4 числові характеристики:
* `sepal_length` — довжина чашолистка
* `sepal_width` — ширина чашолистка
* `petal_length` — довжина пелюстки
* `petal_width` — ширина пелюстки

Цільова змінна:
* `species` — вид квітки (категоріальна змінна)

# Завдання (за етапами EDA)
1. Ознайомлення з даними
* Вивести перші 5 рядків
* Визначити:
  * кількість рядків і стовпців
  * назви колонок
  * типи даних
* Питання:
  * Скільки об’єктів у датасеті?
  * Скільки ознак?
2. Розуміння змінних
* Визначити:
  * числові змінні
  * категоріальні змінні
  * Визначити "цільову змінну"
* Питання:
  * Яка змінна є цільовою?
  * Які змінні описують об’єкти?
3. Перевірка пропущених значень
* Перевірити наявність пропусків
* Питання:
  * Чи є пропущені значення?
  * Якщо є — що з ними робити?
4. Описова статистика
* Вивести статистику для числових змінних
* Питання:
  * Яка ознака має найбільше середнє значення?
  * Чи є сильні відмінності між ознаками?
5. Аналіз категоріальних змінних
* Визначити кількість об’єктів кожного класу
* Питання:
  * Чи збалансований датасет (чи класи розподілені приблизно порівну)?
6. Аналіз розподілів
* Побудувати гістограми для всіх числових змінних
* Побудувати boxplot
* Питання:
  * Чи є викиди?
  * Які розподіли (симетричні чи ні)?
7. Аналіз цільової змінної
* Візуалізувати розподіл класів
* Питання:
  * Який клас найчастіший?
8. Біваріантний аналіз
* Побудувати:
  * scatterplot між ознаками
  * boxplot (ознака vs клас)
* Питання:
  * Які ознаки найкраще розділяють класи?
9. Кореляційний аналіз
* Побудувати кореляційну матрицю
* Питання:
  * Які змінні сильно корелюють?
  * Чи є надлишкові ознаки?
10. Зведені таблиці (Pivot Tables)
* Побудувати середні значення ознак по класах
* Питання:
  * Як відрізняються класи між собою?
11. Виявлення закономірностей
* Описати:
  * які ознаки найважливіші
  * як відрізняються класи
12. Висновки
* Написати висновки:
  * про структуру даних
  * про залежності

In [3]:
## 1. Ознайомлення з даними

import numpy as np
import pandas as pd

# Input data
n_first_rows = 5

# Solution
display(Markdown(f"""
### 1. Ознайомлення з даними
"""))

display(Markdown(f"""
---
Виведення перших {n_first_rows} рядків:
"""), iris.head(n_first_rows))

n_columns = iris.columns.size
n_rows = iris.index.size
columns_df = pd.DataFrame(iris.columns, columns=["Columns"])
types_df = pd.DataFrame(iris.dtypes, columns=["Types"])

display(Markdown(f"""
---
Параметри таблиці:
* Кількість рядків: {iris.index.size}
* Кількість стовпців: {iris.columns.size}
"""))

display(Markdown(f"""
* Назви колонок:
"""), columns_df.style.hide(axis='index'))

display(Markdown(f"""
* Типи даних:
"""), types_df)

col_num_type = iris.select_dtypes(include=np.number).columns
col_cat_type = iris.select_dtypes(exclude=np.number).columns

display(Markdown(f"""
---
Відповіді:
* Кількість об’єктів у датасеті: {iris.index.size}
* Кількість числових ознак: {col_num_type.size}
* Кількість цільових ознак: {col_cat_type.size}
---
"""))


### 1. Ознайомлення з даними



---
Виведення перших 5 рядків:


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa



---
Параметри таблиці:
* Кількість рядків: 150
* Кількість стовпців: 5



* Назви колонок:


Columns
sepal_length
sepal_width
petal_length
petal_width
species



* Типи даних:


,Types
sepal_length,float64
sepal_width,float64
petal_length,float64
petal_width,float64
species,str



---
Відповіді:
* Кількість об’єктів у датасеті: 150
* Кількість числових ознак: 4
* Кількість цільових ознак: 1
---


In [11]:
## 2. Розуміння змінних

import numpy as np
import pandas as pd

# Input data

# Solution
col_num_type_df = pd.DataFrame(col_num_type, columns=["Numeric variables"])
col_cat_type_df = pd.DataFrame(col_cat_type, columns=["Categorical variables"])

display(Markdown(f"""
### 2. Розуміння змінних
"""))

display(Markdown(f"""
---
Визначення:
* Числові змінні:
"""), col_num_type_df.style.hide(axis='index'))

cat_types = iris.select_dtypes(exclude=['str']).columns
col_targ_type_df = pd.DataFrame(col_cat_type, columns=["Target variables"])

display(Markdown(f"""
* Категоріальні змінні:
"""), col_cat_type_df.style.hide(axis='index'))

display(Markdown(f"""
* Цільова зміна:
"""), col_targ_type_df.style.hide(axis='index'))


display(Markdown(f"""
**Примітка**

> Числові змінні - це змінні, які відображають характеристики обʼєкта, його результати, величину впливу на нього тощо. Аналогія: поля ініціалізованого об'єкта.

> Категоріальні змінні - це змінні, що описують обʼєкт відносячи його до певної групи, або ж це групи, поділені за певними діапазонами ознак. Аналогії: 1. бази даних SQL, в якій дочірня таблиця містить у собі зовнішній ключ (подібно до назви групи) і використовується в мастер-таблиці. 2. Enum (типи значень в Java, C#), які несуть у собі певне пояснення у відмінності, або ж як interface (інтерфейс в Java, C#) або абстрактний клас в Python, де при наслідуванні клас наслідує поведінку та деякі характеристики, якщо це передбачено.

> Цільові змінні - це змінні, які описують очікуваний вихідний/прогнозований результат залежно від ознак, тобто назва групи/категорії. Якщо обрати одну з категоріальних змінних і розглянути саме її структуру, то в цьому об'єктові представлення/назва/мітка/ID і буде цільовою змінною. Аналогія: клас ініціалізованого обʼєкта з усіма необхідними полями та наслідуванням від необхідних абстрактних класів.

> Структура: вхідні параметри — це числові й категоріальні змінні, а вихідний результат - цільова змінна. Наприклад: ```def function(*числові та категоріальні змінні) -> цільові змінні:```
"""))



### 2. Розуміння змінних



---
Визначення:
* Числові змінні:


Numeric variables
sepal_length
sepal_width
petal_length
petal_width



* Категоріальні змінні:


Categorical variables
species



* Цільова зміна:


Target variables
species



**Примітка**

> Числові змінні - це змінні, які відображають характеристики обʼєкта, його результати, величину впливу на нього тощо. Аналогія: поля ініціалізованого об'єкта.

> Категоріальні змінні - це змінні, що описують обʼєкт відносячи його до певної групи, або ж це групи, поділені за певними діапазонами ознак. Аналогії: 1. бази даних SQL, в якій дочірня таблиця містить у собі зовнішній ключ (подібно до назви групи) і використовується в мастер-таблиці. 2. Enum (типи значень в Java, C#), які несуть у собі певне пояснення у відмінності, або ж як interface (інтерфейс в Java, C#) або абстрактний клас в Python, де при наслідуванні клас наслідує поведінку та деякі характеристики, якщо це передбачено.

> Цільові змінні - це змінні, які описують очікуваний вихідний/прогнозований результат залежно від ознак, тобто назва групи/категорії. Якщо обрати одну з категоріальних змінних і розглянути саме її структуру, то в цьому об'єктові представлення/назва/мітка/ID і буде цільовою змінною. Аналогія: клас ініціалізованого обʼєкта з усіма необхідними полями та наслідуванням від необхідних абстрактних класів.

> Структура: вхідні параметри — це числові й категоріальні змінні, а вихідний результат - цільова змінна. Наприклад: ```def function(*числові та категоріальні змінні) -> цільові змінні:```
